# Company Policy Assistant — Simple RAG in Google Colab

A plain Retrieval-Augmented Generation (RAG) app, no agent or tool-calling —
just three steps every time: **retrieve** the matching PDF chunks, **augment**
a prompt with them, **generate** an answer with a Hugging Face model.

Everything here is Hugging Face + LangChain, end to end — the embedding model
AND the answer-writing model both run locally inside this Colab session. No
API key, no account, no external LLM service.

**Run the cells below in order, top to bottom.** Colab resets its files every
time the runtime disconnects, so if you come back after a break, re-run from
Cell 1.

You'll need:
- Your `company_policy.pdf` (upload it in Cell 2)
- Nothing else — no API key, no sign-up

A note on speed: the LLM (Cell 5's `app.py`) runs on whatever hardware Colab
gave you. On the free CPU runtime, each answer can take 30–90 seconds — that's
normal. For much faster answers, switch to a GPU first: **Runtime → Change
runtime type → T4 GPU** (still free), then run the cells from the top.


## 1. Install dependencies

In [1]:
!pip install streamlit langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf pdf2image pytesseract transformers torch accelerate --quiet
!apt-get -qq install -y tesseract-ocr poppler-utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Selecting previously unselected package poppler-utils.
(Reading database ... 122809 files and direc

## 2. Upload your policy PDF

Running this cell opens a file picker — select your `company_policy.pdf`.
(If your file has a different name, either rename it to `company_policy.pdf`
when uploading, or change `PDF_FILE` in the `build_index.py` cell below.)


In [2]:
from google.colab import files
uploaded = files.upload()


Saving company_policy.pdf to company_policy.pdf


## 3. Write `build_index.py`

This script reads the PDF (with OCR fallback for scanned pages), splits it
into chunks, embeds them, and saves a `faiss_index/` folder — the searchable
database the app reads from. It does **not** use an LLM or agent; it just
prepares data.


In [6]:
%%writefile build_index.py
"""
STEP 1 of 2 — Build the vector database from the company policy PDF
=====================================================================
This is a ONE-TIME (or "whenever the PDF changes") script. Run it once,
and it creates a folder called faiss_index/ on disk. The Streamlit app
(app.py) then just LOADS that folder — it never re-reads the PDF itself.

What this script does, in plain terms:
    1. Reads company_policy.pdf, page by page.
    2. If a page has real text, it uses that text directly.
       If a page looks like a scanned image (little/no text), it runs
       OCR (Tesseract) on that page instead, so scanned policy PDFs
       still work.
    3. Chops all that text into small overlapping "chunks" (~500
       characters each) — this is what actually gets searched later.
    4. Turns every chunk into a vector (a list of numbers that captures
       its meaning) using a free local embedding model.
    5. Stores all those vectors in a FAISS index (a fast similarity
       search database) and saves it to disk as faiss_index/.

Nothing here uses an LLM, an agent, or the internet at question-answer
time — this script's ONLY job is to prepare the searchable database.
The actual "answer the user's question" logic lives in app.py.

SETUP:
    pip install langchain-community langchain-text-splitters langchain-huggingface \
                faiss-cpu pypdf pdf2image pytesseract --break-system-packages

    You also need two system tools (not python packages) for OCR:
        Ubuntu/Debian:  sudo apt-get install tesseract-ocr poppler-utils
        Mac:            brew install tesseract poppler
        Windows:        install Tesseract:  https://github.com/UB-Mannheim/tesseract/wiki
                         install poppler:    https://github.com/oschwartz10612/poppler-windows

RUN THIS WITH:
    python build_index.py

Put company_policy.pdf in the SAME folder as this script before running it.
"""

from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from pdf2image import convert_from_path
import pytesseract

# ============================================================
# CONFIGURATION — change these two lines if your file/folder names differ
# ============================================================

PDF_FILE = "company_policy.pdf"
DB_PATH = "faiss_index"

# A page is treated as "scanned / needs OCR" if normal text extraction
# returns fewer than this many characters.
MIN_TEXT_LENGTH_THRESHOLD = 20


# ============================================================
# 1. LOAD PDF — TEXT FIRST, OCR FALLBACK PER PAGE
# ============================================================

def load_pdf_with_ocr_fallback(pdf_path: str) -> list[Document]:
    print("Reading PDF (text layer first)...")

    # Try normal text extraction for every page
    text_loader = PyPDFLoader(pdf_path)
    text_pages = text_loader.load()  # one Document per page, in order

    # Render every page as an image ONCE (needed only for pages that fail the text check)
    print("Rendering pages as images (for OCR fallback if needed)...")
    page_images = convert_from_path(pdf_path)

    final_documents = []
    ocr_page_count = 0

    for i, page_doc in enumerate(text_pages):
        extracted_text = page_doc.page_content.strip()

        if len(extracted_text) >= MIN_TEXT_LENGTH_THRESHOLD:
            # Real text layer present and usable — keep as is
            final_documents.append(page_doc)
        else:
            # Looks scanned/empty — OCR this specific page
            ocr_page_count += 1
            print(f"  Page {i + 1}: no usable text layer, running OCR...")
            ocr_text = pytesseract.image_to_string(page_images[i])

            ocr_doc = Document(
                page_content=ocr_text,
                metadata={**page_doc.metadata, "ocr_applied": True},
            )
            final_documents.append(ocr_doc)

    print(f"Total pages: {len(final_documents)}  |  OCR applied to: {ocr_page_count} page(s)")
    return final_documents


# ============================================================
# 2-5. SPLIT -> EMBED -> BUILD FAISS -> SAVE
# ============================================================

def build_index():
    if not Path(PDF_FILE).exists():
        raise FileNotFoundError(
            f"Could not find '{PDF_FILE}'. Put company_policy.pdf in the same "
            f"folder as build_index.py, or change PDF_FILE at the top of this script."
        )

    documents = load_pdf_with_ocr_fallback(PDF_FILE)

    print("Splitting PDF into chunks...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(documents)
    print("Number of chunks:", len(chunks))

    print("Loading embedding model (first run downloads it, ~90MB, one time)...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    print("Creating FAISS database...")
    vector_db = FAISS.from_documents(chunks, embeddings)

    vector_db.save_local(DB_PATH)

    print("\n======================================")
    print("FAISS DATABASE CREATED")
    print("======================================")
    print("Vectors:", vector_db.index.ntotal)
    print("Vector dimensions:", vector_db.index.d)
    print("\nSaved to:", Path(DB_PATH).absolute())
    print("\nNext step: run  streamlit run app.py")


if __name__ == "__main__":
    build_index()


Overwriting build_index.py


## 4. Build the vector database (run once per session)

In [7]:
!python build_index.py


/content/build_index.py:43: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Reading PDF (text layer first)...
Rendering pages as images (for OCR fallback if needed)...
Total pages: 2  |  OCR applied to: 0 page(s)
Splitting PDF into chunks...
Number of chunks: 4
Loading embedding model (first run downloads it, ~90MB, one time)...
modules.json: 100% 349/349 [00:00<00:00, 1.13MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 459kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 22.2MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 194kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.38MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:00<00:00, 130MB/s, 7.44MB/s  ]
model.safetensors: downloading bytes

## 5. Write `app.py`

The Streamlit app itself: for every question it does retrieve → augment →
generate, in that order, with no agent deciding anything. The LLM
(`Qwen/Qwen2.5-1.5B-Instruct`) downloads automatically the first time it
runs — no key or sign-up needed.


In [8]:
%%writefile app.py
"""
STEP 2 of 2 — Streamlit chat app: retrieve from the PDF, then ask the LLM
============================================================================
This is the SIMPLE version on purpose: no agent, no tools, no "the LLM
decides what to do". It always does the exact same 3 steps, in order,
for every question:

    1. RETRIEVE  — search faiss_index/ (built by build_index.py) for the
                    3 chunks of the PDF that best match the question.
    2. AUGMENT   — paste those 3 chunks into a prompt, along with the
                    question, so the LLM has the actual policy text
                    in front of it.
    3. GENERATE  — run that prompt through a Hugging Face model, right
                    here, and show the answer in the chat. The model is
                    told to use the policy text only when it's actually
                    relevant, and fall back to its own general knowledge
                    otherwise — so it stays a normal, useful chatbot
                    instead of refusing anything outside the PDF.

That's it. Retrieval always runs the same way every time — there's no
agent deciding whether to search or which tool to call. This is the same
"Retrieval-Augmented Generation" pattern from the workshop slides, just
written out step by step instead of hidden inside an agent — easier to
read, easier to explain, easier to debug.

Everything in this app is Hugging Face + LangChain, end to end — the
embedding model AND the answer-writing model both run locally, in this
Colab notebook. No API key, no account, no external LLM service.

SETUP:
    pip install streamlit langchain-huggingface langchain-community faiss-cpu transformers torch accelerate --break-system-packages

RUN THIS WITH:
    streamlit run app.py

(Do NOT run it with `python app.py` — Streamlit apps must be launched
with the `streamlit run` command.)

IMPORTANT: run build_index.py FIRST. This app only READS the
faiss_index/ folder it creates — it does not build it.

A NOTE ON SPEED: the LLM below runs on whatever hardware Colab gave you.
On a free CPU runtime, each answer can take 30-90 seconds — that's normal,
not a bug. For much faster answers, switch to a GPU before running this
app: Runtime -> Change runtime type -> T4 GPU (still free).
"""

from pathlib import Path

import streamlit as st
from transformers import pipeline

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


# ============================================================
# CONFIGURATION
# ============================================================

DB_PATH = "faiss_index"

# A small, open, instruction-tuned model — no gated access, no HF token
# needed. Small enough to run (slowly) on a free Colab CPU, and much
# faster if you switch that Colab runtime to a T4 GPU.
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


# ============================================================
# 1. LOAD THE VECTOR DATABASE (once per session, cached)
# ============================================================
# @st.cache_resource = "run this once, keep the result in memory, hand
# back the same object on every rerun." Without it, Streamlit would
# reload the embedding model and the whole FAISS index on every single
# message, since Streamlit reruns the script top-to-bottom each time.

@st.cache_resource(show_spinner="Loading the policy database...")
def load_vector_db():
    if not Path(DB_PATH).exists():
        st.error(
            f"Couldn't find the '{DB_PATH}' folder. Run `python build_index.py` "
            f"first — it reads company_policy.pdf and creates this folder."
        )
        st.stop()

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return FAISS.load_local(
        DB_PATH,
        embeddings,
        allow_dangerous_deserialization=True,
    )


# ============================================================
# 1b. LOAD THE LOCAL LLM (once per session, cached)
# ============================================================
# This downloads LLM_MODEL the first time it runs (a few GB — give it a
# couple of minutes) and keeps it in memory after that. This is the part
# that replaces a call to an external API like Groq or OpenAI: the model
# itself lives inside this Colab session.

@st.cache_resource(show_spinner="Downloading and loading the local LLM (first run only, can take a few minutes)...")
def load_llm_pipeline():
    return pipeline(
        "text-generation",
        model=LLM_MODEL,
        torch_dtype="auto",
        device_map="auto",
    )


# ============================================================
# 2. RETRIEVE — plain similarity search, no agent involved
# ============================================================

def retrieve_context(vector_db, question: str, k: int = 3) -> str:
    matching_chunks = vector_db.similarity_search(question, k=k)
    if not matching_chunks:
        return ""
    return "\n---\n".join(chunk.page_content for chunk in matching_chunks)


# ============================================================
# 3. AUGMENT + GENERATE — build the prompt ourselves, run it through the
#    local Hugging Face pipeline directly
# ============================================================

SYSTEM_PROMPT = """You are the company's assistant — both a Policy Assistant and a normal helpful chatbot.

Some retrieved policy text is pasted below with every question, whether or not it's actually relevant — it comes from an automatic search, not a human. Decide for yourself if it applies:

- If the question is about company policy (leave, WFH, expenses, IT, HR rules, etc.) AND the retrieved text below actually answers it, answer using ONLY that text. Never invent or guess policy details.
- If the question is about company policy but the retrieved text does NOT contain a relevant answer, say so honestly and suggest contacting HR — do not guess.
- If the question has nothing to do with company policy (general knowledge, casual conversation, coding help, etc.), ignore the retrieved text completely and just answer normally, like any helpful assistant would.
- Be concise.
"""


def ask_llm(llm_pipeline, context: str, question: str) -> str:
    user_prompt = f"""Retrieved policy text (automatic search result — may or may not be relevant to this question):
\"\"\"
{context if context else "(No matching policy text was found.)"}
\"\"\"

Question: {question}

Follow the system rules: use the retrieved text only if it's actually a relevant policy question; otherwise answer normally."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    output = llm_pipeline(messages, max_new_tokens=300, do_sample=False)
    generated = output[0]["generated_text"]

    # Modern transformers chat pipelines return the full message list back
    # (system + user + the new assistant reply) — grab just the reply.
    # Older versions may return a plain string instead, so handle both.
    if isinstance(generated, list):
        return generated[-1]["content"].strip()
    return str(generated).strip()


# ============================================================
# 4. PAGE SETUP
# ============================================================

st.set_page_config(page_title="Policy Assistant", page_icon="\U0001F4CB")
st.title("Company Policy Assistant")
st.caption("Ask about company policy, or ask me anything else — I'll use the PDF when it's relevant, and my own knowledge otherwise.")

with st.sidebar:
    st.subheader("Running fully locally")
    st.caption(f"No API key needed — both the search and the answer-writing model ({LLM_MODEL}) run right here in this Colab session.")
    st.divider()
    st.markdown(
        "**How this works:**\n\n"
        "1. Your question is compared against chunks of the policy PDF (retrieval) — this always happens.\n"
        "2. The 3 best-matching chunks are pasted into a prompt (augmentation).\n"
        "3. The LLM decides: if they're actually relevant, it answers strictly from them; if not, it answers from its own general knowledge instead (generation)."
    )
    st.divider()
    st.caption("On a free Colab CPU, answers can take 30-90 seconds. Switch to a T4 GPU (Runtime -> Change runtime type) for faster responses.")

vector_db = load_vector_db()
llm_pipeline = load_llm_pipeline()


# ============================================================
# 5. CHAT HISTORY
# ============================================================
# st.session_state persists across reruns WITHIN one browser session —
# this replaces the Python list you'd build up inside a while-True loop
# in a plain terminal script.

if "messages" not in st.session_state:
    st.session_state.messages = []

# Redraw all previous messages on every rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])


# ============================================================
# 6. CHAT INPUT — replaces: question = input("You: ")
# ============================================================

user_question = st.chat_input("Ask a question about company policy...")

if user_question:

    # Show the user's message immediately
    with st.chat_message("user"):
        st.markdown(user_question)
    st.session_state.messages.append({"role": "user", "content": user_question})

    # The three RAG steps, called directly — no agent deciding anything
    with st.chat_message("assistant"):
        with st.spinner("Searching the policy PDF..."):
            context = retrieve_context(vector_db, user_question, k=3)

        with st.expander("Show the policy text used to answer this"):
            st.text(context if context else "No matching text was found.")

        with st.spinner("Thinking... (can take a while on a free CPU)"):
            answer = ask_llm(llm_pipeline, context, user_question)

        st.markdown(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})


Writing app.py


## 6. Launch Streamlit in the background

The first question you ask will be slow — that's the LLM downloading (a few
GB, one time) and loading into memory. After that it stays loaded for the
rest of the session.


In [10]:
!streamlit run app.py &>/content/logs.txt &


## 7. Expose it with a public tunnel

This prints a URL like `https://short-words-123.loca.lt` — click it to open
the app in a new tab.


In [ ]:
!npx localtunnel --port 8501


⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋your url is: https://brave-feet-smell.loca.lt


## 8. Get the tunnel password

The tunnel page asks for a password before showing your app — it's just your
Colab runtime's public IP address. Run this cell, copy the output, and paste
it into the "Tunnel Password" field on that page.


In [ ]:
!curl ipv4.icanhazip.com


## Troubleshooting

- **"Couldn't find the 'faiss_index' folder"** in the app → Cell 4 didn't run
  successfully, or the runtime restarted since you ran it. Re-run Cells 3–4.
- **Tunnel link doesn't load** → Cell 6 (Streamlit) may still be starting;
  wait ~10 seconds and refresh, or check `/content/logs.txt` for errors:
  `!cat /content/logs.txt`
- **New tunnel URL each time** → this is expected; re-run Cell 7 whenever you
  restart Streamlit and share the new link.
- **Answers take a long time / app looks frozen** → normal on a free CPU
  runtime (30–90 seconds per answer, longer on the very first question while
  the model loads). Switch to a T4 GPU (Runtime → Change runtime type) and
  re-run from Cell 1 for much faster responses.
- **"Out of memory" / session crashes when loading the model** → the free
  Colab CPU runtime occasionally runs low on RAM for larger downloads.
  Runtime → Restart session, then re-run from Cell 1; switching to a GPU
  runtime usually has more headroom too.
